In [3]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class SpacecraftEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # 状态：13维
        self.state_dim = 13
        self.action_dim = 6

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.state_dim,), dtype=np.float32)

        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(self.action_dim,), dtype=np.float32)

        self.dt = 0.05
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)  # ⭐ 很关键

        r = np.random.randn(3) * 100
        v = np.zeros(3)
        q = np.array([1, 0, 0, 0])
        w = np.zeros(3)

        self.state = np.concatenate([r, v, q, w])

        return self.state, {}   # Gymnasium 必须返回 tuple


    def step(self, action):
        r = self.state[0:3]
        v = self.state[3:6]
        q = self.state[6:10]
        w = self.state[10:13]

        F = action[0:3]
        M = action[3:6]

        # ===== 简化动力学（你后面可以换成真实CW+刚体）=====
        v = v + F * self.dt
        r = r + v * self.dt

        w = w + M * self.dt
        # 姿态先简化（不做严格四元数积分）
        q = q  # TODO: 换成真实 quaternion dynamics

        self.state = np.concatenate([r, v, q, w])

        # ===== reward（核心）=====
        pos_error = np.linalg.norm(r)
        att_error = 1 - q[0]  # 标量部分

        # 约束（模拟你的 f1, f2）
        constraint_penalty = max(0, -r[0])  # 举例：不能进入某区域

        reward = - (10*pos_error + 5*att_error + 20*constraint_penalty)

        done = pos_error < 0.1

        return self.state, reward, done, {}


In [4]:
from stable_baselines3 import PPO

env = SpacecraftEnv()

model = PPO("MlpPolicy", env, verbose=1)

model.learn(total_timesteps=100000)

model.save("spacecraft_rl")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


ValueError: not enough values to unpack (expected 5, got 4)